# AIC 2026 - Tinh Chỉnh Prompt & Kiến Trúc Captioning Với Qwen3-VL-8B-Instruct Trên Kaggle

Notebook này được tối ưu hóa cho bài toán **Fine-tuning Prompt & Đánh Giá Chất Lượng Trực Quan** mô hình **`Qwen/Qwen3-VL-8B-Instruct`** trên **12 Testcases Chuẩn** trước khi đưa vào chạy hàng loạt:

### 🎯 CÁC NGUYÊN TẮC CẢI TIẾN TRỌNG TÂM CỦA PROMPT V2:
1. **Cài đặt `max_new_tokens = 512`:** Cho phép mô hình diễn giải sâu sắc, đầy đủ mọi chi tiết thị giác và cấu trúc không gian mà không bị ngắt cụt câu.
2. **Ngữ cảnh nền tảng (Context-only):** Sử dụng `Title Video`, `Lời thoại ASR` và khả năng đọc `OCR` làm **kiến thức nền** để nhận diện chính xác danh tính nhân vật, món ăn, sự kiện, địa điểm.
3. **CẤM CHÉP LẠI ASR & OCR:** Tuyệt đối không trích dẫn câu thoại ASR (như *cô gái nói 'tốt lắm'*) và không chép chữ OCR/logo (như *chữ 'món ngon'*, *logo HTV*). Toàn bộ ASR và OCR đã có kênh Search chuyên biệt phụ trách; caption VLM phải tập trung **100% vào mô tả thị giác thuần túy**.
4. **Cơ chế mô tả thích ứng (Chuyển động vs Cảnh tĩnh):**
   - **Cảnh có chuyển động:** Trọng tâm miêu tả hành động chính, sự di dời của người, vật thể và tác động qua lại.
   - **Cảnh tĩnh:** Tập trung miêu tả sâu cấu trúc thực thể, cách sắp đặt, bố cục đồ vật, sơ đồ/bài giảng, màu sắc và nội dung biểu đạt.
5. **Hiển thị trực tiếp trực quan trên màn hình:** Định dạng Markdown/HTML rõ ràng từng testcase để người dùng đối chiếu, đánh giá và tinh chỉnh prompt ngay trên Kaggle.

--- 
## BƯỚC 1: Kiểm Tra Tài Nguyên GPU Trên Kaggle
*(Khuyến nghị: Chọn **GPU T4 x2 (Dual T4)** trong phần Settings để model 8B chạy 100% trên 32GB VRAM)*

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Số lượng GPU phát hiện: {device_count}")
for i in range(device_count):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)} | VRAM: {round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)} GB")
if device_count >= 2:
    print("🔥 Tuyệt vời! Dual GPU phát hiện, model 8B sẽ được tự động chia đều để chạy mượt mà không nghẽn RAM!")
print("=" * 60)

--- 
## BƯỚC 2: Cấu Hình Dataset & Tự Động Ánh Xạ 12 Testcases

In [ ]:
import os
import re
import json
from datetime import datetime

# ==============================================================================
# 🎯 1. CẤU HÌNH ĐƯỜNG DẪN DATASET TRÊN KAGGLE (ĐỂ TRỐNG "" ĐỂ TỰ ĐỘNG DÒ TÌM):
# ==============================================================================
VIDEO_DIR = ""       # Folder chứa video shot (.mp4)
MEDIA_INFO_DIR = ""  # Folder chứa metadata {video_id}.json
ASR_DIR = ""         # Folder chứa {video_id}.json lời thoại
OUTPUT_DIR = "/workspace/output" if os.path.exists("/workspace") else "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_JSONL = os.path.join(OUTPUT_DIR, "qwen3vl_8b_finetuned_captions.jsonl")
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "qwen3vl_8b_finetuned_captions.json")
RESUME_CHECKPOINT = True

# ==============================================================================
# 📋 2. DANH SÁCH 12 TESTCASES CỤ THỂ ĐỂ FINETUNE:
# ==============================================================================
TARGET_TEST_FILENAMES = [
    "L26_V480_shot_0060_03m04s_03m08s.mp4",  # 1. Chảo gừng dứa cá dứa (Ẩm thực)
    "L24_V013_shot_0063_07m06s_07m12s.mp4",  # 2. Con lân đỏ cận cảnh (Lễ hội)
    "L25_V060_shot_0197_18m52s_19m00s.mp4",  # 3. Slide Địa lý dân cư (Cảnh tĩnh / Giáo dục)
    "L26_V254_shot_0004_00m12s_00m19s.mp4",  # 4. Cầm nghêu kho tộ (Ẩm thực)
    "L23_V017_shot_0017_01m41s_01m43s.mp4",  # 5. Đua xe đạp cự ly 0.8km (Thể thao)
    "L24_V013_shot_0062_06m59s_07m06s.mp4",  # 6. Con lân đỏ trung cảnh (Lễ hội)
    "L23_V017_shot_0016_01m36s_01m41s.mp4",  # 7. Đua xe đạp cự ly 0.9km (Thể thao)
    "L23_V010_shot_0003_00m20s_00m25s.mp4",  # 8. Đua xe đạp chặng 10 tại Huế (Thể thao)
    "L26_V074_shot_0007_00m28s_00m32s.mp4",  # 9. Bàn lẩu Thái và hoa toàn cảnh (Cảnh tĩnh ẩm thực)
    "L26_V074_shot_0005_00m16s_00m23s.mp4",  # 10. Bó hoa và tôm sống (Cảnh tĩnh ẩm thực)
    "L26_V074_shot_0006_00m23s_00m28s.mp4",  # 11. Bó hoa và thịt heo (Cảnh tĩnh ẩm thực)
    "L25_V014_shot_0179_15m28s_15m30s.mp4",  # 12. Bài giảng Vật lý lượng tử công thức lambda (Cảnh tĩnh bài tập)
]

# ==============================================================================
# 📂 3. TỰ ĐỘNG DÒ TÌM THƯ MỤC SIÊU TỐC (0.01 GIÂY) TRÊN VAST.AI / KAGGLE / LOCAL:
# ==============================================================================
input_candidates = ["/workspace/data", "/workspace", "/kaggle/input", "."]
input_base = next((p for p in input_candidates if os.path.exists(p)), ".")

def find_dir_fast(base_path, target_names):
    """Dò tìm nhanh thư mục mà KHÔNG duyệt đệ quy vào 873 subfolder video."""
    if not os.path.exists(base_path):
        return ""
    for root, dirs, _ in os.walk(base_path):
        # Pruning: Loại bỏ duyệt sâu vào các folder video Lxx_Vxxx để tránh nghẽn I/O
        dirs[:] = [d for d in dirs if not (d.startswith('L') and '_V' in d)]
        for d in dirs:
            if d.lower() in target_names:
                return os.path.join(root, d)
    return ""

if not VIDEO_DIR or not os.path.exists(VIDEO_DIR):
    VIDEO_DIR = find_dir_fast(input_base, ["cut_videos", "cut_video", "video_shots", "videos"])
    if VIDEO_DIR:
        print(f"💡 [Tự động phát hiện] Gán VIDEO_DIR = {VIDEO_DIR}")

if not MEDIA_INFO_DIR or not os.path.exists(MEDIA_INFO_DIR):
    MEDIA_INFO_DIR = find_dir_fast(input_base, ["media-info", "media_info", "mediainfo"])
    if MEDIA_INFO_DIR:
        print(f"💡 [Tự động phát hiện] Gán MEDIA_INFO_DIR = {MEDIA_INFO_DIR}")

if not ASR_DIR or not os.path.exists(ASR_DIR):
    ASR_DIR = find_dir_fast(input_base, ["asr"])
    if ASR_DIR:
        print(f"💡 [Tự động phát hiện] Gán ASR_DIR = {ASR_DIR}")

# ==============================================================================
# 🔍 4. TÌM KIẾM TRỰC TIẾP 12 TESTCASES (TRUY XUẤT THẲNG THEO VIDEO_ID TRONG 0.001s):
# ==============================================================================
test_shot_paths = []
print("=" * 75)
print("🔍 TÌM KIẾM 12 TESTCASES TRONG DATASET KAGGLE:")

for idx, fname in enumerate(TARGET_TEST_FILENAMES, 1):
    parts = fname.split('_')
    vid = f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else ""
    
    # 1. Kiểm tra trực tiếp theo cấu trúc chuẩn: cut_videos/{video_id}/{fname}
    direct_path = os.path.join(VIDEO_DIR, vid, fname) if VIDEO_DIR and vid else ""
    flat_path = os.path.join(VIDEO_DIR, fname) if VIDEO_DIR else ""
    
    found_path = ""
    if direct_path and os.path.exists(direct_path):
        found_path = direct_path
    elif flat_path and os.path.exists(flat_path):
        found_path = flat_path
    else:
        # Fallback dò tìm chỉ trong folder của video_id đó nếu cần
        for root, dirs, files in os.walk(input_base):
            dirs[:] = [d for d in dirs if not (d.startswith('L') and '_V' in d and d != vid)]
            if fname in files:
                found_path = os.path.join(root, fname)
                break
                
    if found_path:
        test_shot_paths.append(found_path)
        print(f"  [{idx:02d}/12] ✅ Tìm thấy: {fname}")
    else:
        print(f"  [{idx:02d}/12] ❌ Không thấy: {fname}")

print(f"\n👉 Đã sẵn sàng chạy: {len(test_shot_paths)}/{len(TARGET_TEST_FILENAMES)} testcases")
print(f"👉 File lưu kết quả: {OUTPUT_JSONL}")
print("=" * 75)

--- 
## BƯỚC 3: Cài Đặt & Nâng Cấp Thư Viện Cần Thiết

In [ ]:
# 1. Nâng cấp transformers, accelerate, qwen-vl-utils (Dùng %pip để cài trực tiếp vào đúng Kernel/Venv đang mở)
print("📦 Đang cài đặt thư viện cần thiết vào Kernel Python...")
%pip install -q "transformers>=4.49.0" "accelerate>=1.2.0" "qwen-vl-utils[decord]>=0.0.8" huggingface_hub

# 2. BYPASS AN TOÀN TRIỆT ĐỂ LỖI 'operator torchvision::nms does not exist':
# Khóa file _meta_registrations của torchvision ở tầng Python module để ngăn nạp fake nms/qnms.
# Phương pháp này 100% thuần Python, không can thiệp C++ Dispatcher, an toàn tuyệt đối không làm chết kernel.
import sys
import types
if 'torchvision._meta_registrations' not in sys.modules:
    sys.modules['torchvision._meta_registrations'] = types.ModuleType('torchvision._meta_registrations')

import torch
import torchvision
import transformers
import qwen_vl_utils

print("\n" + "=" * 65)
print("✅ TẤT CẢ THƯ VIỆN ĐÃ ĐƯỢC NẠP THÀNH CÔNG VÀ AN TOÀN:")
print(f"  - PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - Torchvision: {torchvision.__version__} (Bypassed nms an toàn)")
print(f"  - Transformers: {transformers.__version__}")
print(f"  - Qwen-VL-Utils: {qwen_vl_utils.__file__}")
print("=" * 65)

--- 
## BƯỚC 4: Cấu Hình Prompt V5 (Ràng Buộc Phương Hướng Vật Lý Cho Camera) & FPS=1.0

👉 **Các điểm nâng cấp trọng tâm từ `prompt_shot_v5`:**
- **Mục 4: Khóa nguồn tên riêng & chống bịa tên:** Chỉ nêu tên người, tên đội, sự kiện khi chính chữ đó hiện trên màn hình và đọc được. ASR & Title chỉ dùng làm ngữ cảnh chính tả, không lấy làm nguồn thông tin thị giác.
- **Ràng buộc phương hướng không gian vật lý cho camera:** Khi camera chuyển động (lia ngang, lia dọc, zoom, bám theo), **bắt buộc** nêu rõ hướng vật lý (*từ trái sang phải / từ phải sang trái / từ trên xuống dưới / từ dưới lên trên / từ trước ra sau / từ xa lại gần*) kết hợp mốc đồ vật: `[chuyển động] [phương hướng] (từ [mốc A] sang [mốc B])`.
- **Video Sampling chuẩn:** `VIDEO_FPS = 1.0` (giữ 100% độ sắc nét pixel gốc, không nén mờ ảnh).
- **Mục 3 Chữ trên màn hình:** Ghi nguyên văn trong ngoặc kép; chữ nhỏ mờ thì dùng 'hoặc', cấm đoán mò tạo chuỗi giả.

In [ ]:
# Cấu hình token đầu ra & Lấy mẫu video chuẩn (FPS=1.0, giữ 100% độ phân giải gốc cho OCR và thực thể)
MAX_NEW_TOKENS = 512
VIDEO_FPS = 1.0
MIN_FRAMES = 4
MAX_FRAMES = 16

SYSTEM_PROMPT = (
    "Bạn là người hỗ trợ phân tích thị giác và lập chỉ mục dữ liệu video phục vụ hệ thống tìm kiếm (Video Retrieval). "
    "Mỗi đoạn video là MỘT CÚ MÁY ĐƠN LIÊN TỤC (single continuous shot, không có chuyển cảnh). "
    "Nhiệm vụ của bạn là quan sát diễn biến, bóc tách thực thể, màu sắc, số lượng và chi tiết thị giác "
    "một cách tự nhiên, chính xác và giàu thông tin nhất. "
    "Chỉ mô tả những gì NHÌN THẤY trong khung hình."
)

def build_user_prompt(asr_text: str, video_title: str = "") -> str:
    """Tạo prompt chuẩn hóa V5: Ràng buộc phương hướng không gian vật lý cho camera motion."""
    title_block = ""
    if video_title and video_title.strip():
        title_block = f'\n[TIÊU ĐỀ VIDEO / CHƯƠNG TRÌNH]:\n"{video_title.strip()}"\n'

    asr_context = (
        f'"{asr_text.strip()}"'
        if asr_text and asr_text.strip()
        else "(Không có lời thoại / Video chỉ có âm thanh hiện trường hoặc nhạc nền)"
    )

    return f"""Hãy phân tích cú máy video này phục vụ tìm kiếm thông tin:
{title_block}
[LỜI THOẠI ASR]:
{asr_context}

HƯỚNG DẪN MÔ TẢ:

1. Trọng tâm chuyển động: Tập trung miêu tả hành động chính, người hoặc vật đang chuyển động và các sự vật bị tác động chuyển động theo. Nếu là cảnh tĩnh (không có chuyển động), tập trung miêu tả nội dung, bố cục và chi tiết các thực thể tĩnh.

2. Màu sắc & Số lượng (Kể cả đặc điểm hiển nhiên): Luôn chủ động gắn kèm màu sắc và số lượng (hoặc khoảng ước lượng nếu có) cho các thực thể/danh từ, ngay cả khi đó là thuộc tính hiển nhiên (ví dụ: củ dền màu đỏ/tím, cọng sả màu xanh/vàng, lòng đỏ trứng màu cam/vàng). Nếu chưa chắc chắn về tên vật thể hoặc màu sắc, dùng 'hoặc' (hoặc '/') để dự đoán gần đúng. Nếu chưa chắc chắn về số lượng, bắt buộc dùng khoảng (ví dụ: 2-3 cái, khoảng hơn 10 con)

3. Chữ trên màn hình: Đọc và ghi lại văn bản quan trọng NHÌN THẤY TRONG KHUNG HÌNH (dòng chữ chèn trên hình, phụ đề, biển hiệu, bảng điện tử, chữ trên logo, chữ trên trang phục hoặc đồ vật) NGUYÊN VĂN, đặt trong ngoặc kép. Chữ nhỏ hoặc mờ thì áp dụng đúng quy ước 'hoặc' ở mục 2: ghi phần đọc chắc, phần không chắc nêu vài khả năng gần đúng hoặc bỏ riêng phần đó. Điều phải tránh là viết ra MỘT dãy số/chuỗi hoàn chỉnh trông như đã đọc chắc chắn trong khi thực ra đang đoán. Với slide hoặc văn bản dài, chỉ ghi tiêu đề và các dòng mang thông tin chính, không chép lại toàn bộ.

4. Giới hạn nguồn thông tin: Caption chỉ được chứa thứ NHÌN THẤY trong khung hình. Tiêu đề video và lời thoại ASR được cấp cho bạn làm bối cảnh, nhưng KHÔNG phải là nguồn để viết caption:
   - Không nêu tên người, tên đội, tên chương trình, tên sự kiện, tên món ăn hay địa danh trừ khi chính chữ đó hiện trên màn hình và bạn đọc được.
   - Không trích lại lời thoại dưới bất kỳ hình thức nào, kể cả khi đang tả một người đang nói.
   - Không mô tả tiêu đề video như thể nó là chữ hiện trên màn hình. Đọc được bao nhiêu trên hình thì ghi bấy nhiêu; đọc không ra thì tả hình dạng ("logo tròn màu đỏ ở góc trên bên trái") rồi dừng lại, KHÔNG lấy tiêu đề video để điền vào.
   Lời nói thường không khớp với hình đang chiếu: phát thanh viên có thể đang nhắc tên một người không có trong khung hình. Không chắc ai là ai thì tả theo đặc điểm nhìn thấy (màu áo, số áo, vị trí), đừng gán tên.

XUẤT ĐẦU RA THEO ĐÚNG 2 MỤC SAU (KHÔNG THÊM LỜI DẪN):

- Chi tiết thị giác & Thực thể: (Miêu tả tự nhiên hành động, chuyển động của nhân vật, trang phục, màu sắc, số lượng nếu có, đạo cụ, đồ vật chính và chữ hiển thị trên màn hình nếu có)

- Góc máy & Không gian: (Cỡ cảnh: cận cảnh/trung cảnh/toàn cảnh. Sau đó BẮT BUỘC nêu chuyển động camera, chọn đúng một trong: tĩnh · lia ngang · lia dọc · zoom vào · zoom ra · bám theo chủ thể · bay trên cao. Camera đổi kiểu giữa chừng thì nêu cả hai theo thứ tự.
  ĐẶC BIỆT KHI CAMERA CÓ CHUYỂN ĐỘNG (lia ngang, lia dọc, zoom, bám theo): BẮT BUỘC nêu rõ PHƯƠNG HƯỚNG KHÔNG GIAN VẬT LÝ (từ trái sang phải / từ phải sang trái / từ trên xuống dưới / từ dưới lên trên / từ trước ra sau / từ xa lại gần / tiến lại gần / lùi ra xa) kết hợp với mốc đồ vật hoặc nhân vật nếu có. Cú pháp chuẩn: [chuyển động] [phương hướng vật lý] (từ [mốc vật thể A] sang [mốc vật thể B]). Ví dụ: 'camera lia ngang từ trái sang phải (từ bó hoa sang đĩa tôm và thịt)', 'camera lia dọc từ trên xuống dưới (từ biển hiệu trên cao xuống bậc thềm vỉa hè)'.
  Cách xác định camera tĩnh: chỉ chọn "tĩnh" khi khung nền KHÔNG đổi trong suốt đoạn. Nếu hậu cảnh trôi qua khung hình, hoặc vật thể đứng yên dịch chuyển trong khung, thì camera đang chuyển động — không phải tĩnh.
  Cuối cùng nêu không gian trong nhà/ngoài trời.)

VÍ DỤ MẪU:

Ví dụ 1:
- Chi tiết thị giác & Thực thể: Hai công nhân vệ sinh mặc áo bảo hộ màu cam và quần xanh đậm đang dùng chổi tre quét lá khô màu nâu trên vỉa hè lát gạch xám. Một người đẩy chiếc xe rác màu xanh lá có 2 bánh, bên trong chứa khoảng 3-4 bao rác màu đen. Phía dưới khung hình có dòng chữ "THU GOM RÁC SINH HOẠT".
- Góc máy & Không gian: Trung cảnh, camera bám theo chủ thể khi hai người di chuyển dọc vỉa hè; không gian ngoài trời trên đường phố có hàng cây hai bên.

Ví dụ 2:
- Chi tiết thị giác & Thực thể: Một chiếc xe buýt màu vàng và trắng đang rẽ trái vào bến, trên kính chắn gió có bảng số tuyến. Khoảng 5-7 hành khách đứng chờ dưới mái che màu xanh dương, trong đó một phụ nữ mặc áo sơ mi trắng đang cầm điện thoại. Biển hiệu tại bến ghi "TRAM DUNG XE BUYT SO 12".
- Góc máy & Không gian: Toàn cảnh, camera lia ngang từ trái sang phải theo hướng xe chạy rồi dừng lại; không gian ngoài trời.
"""

print(f"✅ Đã cấu hình Prompt V4 (Khóa triệt để ASR/Title, Giới hạn slide dài, Camera danh sách đóng)!")

--- 
## BƯỚC 5: Nạp Model Qwen3-VL-8B-Instruct Với device_map='auto'

Model **`Qwen/Qwen3-VL-8B-Instruct`** (~16.5 GB weights). Khi chạy trên Kaggle GPU T4 x2 (Dual T4 - 32GB VRAM), mô hình sẽ tự động chia đều lên 2 GPU, nạp 100% trong VRAM không cần offload CPU!

In [ ]:
import time
import sys
import types
import torch

if 'torchvision._meta_registrations' not in sys.modules:
    sys.modules['torchvision._meta_registrations'] = types.ModuleType('torchvision._meta_registrations')

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MIN_PIXELS = 128 * 28 * 28   # 100,352 pixels
MAX_PIXELS = 360 * 640       # 230,400 pixels

if 'model' in globals() and model is not None and 'processor' in globals() and processor is not None:
    print("⚡ [BỎ QUA NẠP LẠI] Model và Processor đã có sẵn trong VRAM!")
    print("   => Đại ca có thể thoải mái bấm 'Run all below' từ Bước 4 mà không sợ tốn thời gian nạp lại.")
else:
    from transformers import AutoProcessor
    print(f"[1/2] Đang nạp Processor cho model: {MODEL_ID}...")
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS
    )

    print(f"[2/2] Đang nạp Model weights với dtype=bfloat16, device_map='auto'...")
    start_load = time.time()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    candidate_classes = [
        "Qwen3VLForConditionalGeneration",
        "AutoModelForMultimodalLM",
        "AutoModelForImageTextToText"
    ]

    model = None
    loaded_class_name = None

    num_gpus = torch.cuda.device_count()
    max_memory = {i: "10GiB" for i in range(num_gpus)} if num_gpus > 1 else None
    if max_memory:
        print(f"  [Memory Budget]: Cân bằng VRAM đều mỗi GPU tối đa 10GiB (dành buffer 4.5GiB cho KV-cache)...")

    for class_name in candidate_classes:
        if hasattr(transformers, class_name):
            try:
                cls = getattr(transformers, class_name)
                print(f"  Thử nạp với class: {class_name}...")
                model = cls.from_pretrained(
                    MODEL_ID,
                    torch_dtype=dtype,
                    device_map="auto",
                    max_memory=max_memory
                )
                loaded_class_name = class_name
                print(f"  => ✅ NẠP THÀNH CÔNG với class: {class_name} ({round(time.time() - start_load, 2)}s)")
                break
            except Exception as e:
                print(f"  Class {class_name} thất bại: {e}")
                continue

    if model is None:
        raise RuntimeError(f"Không thể nạp model {MODEL_ID}!")

    model.eval()

print("\n--- THỐNG KÊ PHÂN BỔ VRAM SAU KHI NẠP ---")
for i in range(torch.cuda.device_count()):
    alloc_gb = round(torch.cuda.memory_allocated(i) / (1024**3), 2)
    res_gb = round(torch.cuda.memory_reserved(i) / (1024**3), 2)
    print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): Allocated = {alloc_gb} GB | Reserved = {res_gb} GB")
if hasattr(model, "hf_device_map"): 
    print(f"  [Device Map]: Các layers được phân bổ trên thiết bị: {set(model.hf_device_map.values())}")
print("-------------------------------------------")


--- 
## BƯỚC 6: Các Tiện Ích Trích Xuất Metadata & Hàm Suy Luận

In [ ]:
import os
import re
import json
import sys
import types
import torch

if 'torchvision._meta_registrations' not in sys.modules:
    sys.modules['torchvision._meta_registrations'] = types.ModuleType('torchvision._meta_registrations')

from qwen_vl_utils import process_vision_info

def parse_shot_filename(filename: str):
    """Trích xuất video_id, shot_idx, start_sec, end_sec từ tên file shot."""
    stem = os.path.splitext(os.path.basename(filename))[0]
    parts = stem.split('_')
    if len(parts) >= 6:
        video_id = f"{parts[0]}_{parts[1]}"
        try:
            shot_idx = int(parts[3])
        except Exception:
            shot_idx = 0
        def to_sec(ts: str) -> float:
            h_m = re.search(r'(\d+)h', ts)
            m_m = re.search(r'(\d+)m', ts)
            s_m = re.search(r'(\d+)s', ts)
            hours = int(h_m.group(1)) if h_m else 0
            mins = int(m_m.group(1)) if m_m else 0
            secs = int(s_m.group(1)) if s_m else 0
            return float(hours * 3600 + mins * 60 + secs)
        start_sec = to_sec(parts[-2])
        end_sec = to_sec(parts[-1])
        return video_id, shot_idx, start_sec, end_sec
    return stem, 0, 0.0, 0.0


def get_video_title(video_id: str, media_info_dir: str) -> str:
    """Đọc file JSON metadata trong media-info tương ứng với video_id và lấy trường title."""
    if not media_info_dir or not os.path.exists(media_info_dir):
        return ""
    info_path = os.path.join(media_info_dir, f"{video_id}.json")
    if not os.path.exists(info_path):
        return ""
    try:
        with open(info_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data.get("title", "").strip()
    except Exception:
        return ""


def get_asr_context(video_id: str, shot_start: float, shot_end: float, asr_dir: str):
    """Lọc các segment lời thoại ASR giao thoa với khoảng thời gian của shot."""
    if not asr_dir or not os.path.exists(asr_dir):
        return "", 0
    asr_path = os.path.join(asr_dir, f"{video_id}.json")
    if not os.path.exists(asr_path):
        return "", 0
    try:
        with open(asr_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        segments = data if isinstance(data, list) else data.get("segments", [])
        matched_texts = []
        for seg in segments:
            s_start = seg.get("start", 0.0)
            s_end = seg.get("end", 0.0)
            text = seg.get("text", "").strip()
            if not text:
                continue
            if max(s_start, shot_start) < min(s_end, shot_end):
                matched_texts.append(text)
        full_text = " ".join(matched_texts).strip()
        return full_text, len(matched_texts)
    except Exception:
        return "", 0


def caption_single_shot(
    model,
    processor,
    video_path: str,
    asr_text: str = "",
    video_title: str = "",
    fps: float = VIDEO_FPS,
    min_frames: int = MIN_FRAMES,
    max_frames: int = MAX_FRAMES,
    max_new_tokens: int = MAX_NEW_TOKENS,
    repetition_penalty: float = 1.05,
):
    """Sinh caption cho một video shot duy nhất sử dụng Qwen3-VL-8B với Prompt V4."""
    start_time = time.time()
    user_prompt = build_user_prompt(asr_text, video_title)
    video_uri = f"file://{os.path.abspath(video_path)}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_uri,
                    "fps": fps,
                    "min_frames": min_frames,
                    "max_frames": max_frames,
                },
                {"type": "text", "text": user_prompt},
            ],
        },
    ]

    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    vision_out = process_vision_info(messages, return_video_kwargs=True)
    if isinstance(vision_out, tuple) and len(vision_out) == 3:
        image_inputs, video_inputs, video_kwargs = vision_out
    elif isinstance(vision_out, tuple) and len(vision_out) == 2:
        image_inputs, video_inputs = vision_out
        video_kwargs = {}
    else:
        image_inputs, video_inputs = None, vision_out
        video_kwargs = {}

    if isinstance(video_kwargs, dict):
        if "fps" in video_kwargs and isinstance(video_kwargs["fps"], (list, tuple)):
            if len(video_kwargs["fps"]) > 0:
                video_kwargs["fps"] = float(video_kwargs["fps"][0])
            else:
                video_kwargs.pop("fps", None)

    inputs = processor(
        text=[prompt_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=repetition_penalty
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = generated_ids[:, input_len:]
    caption = processor.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return {
        "caption": caption,
        "infer_time_sec": round(time.time() - start_time, 2),
        "input_tokens": int(input_len),
        "generated_tokens": int(generated_tokens.shape[1]),
    }

print("✅ Đã khởi tạo các hàm tiện ích và suy luận!")

--- 
## BƯỚC 7: Chạy Suy Luận & Hiển Thị Trực Quan Để Đánh Giá (Finetune Mode)

In [ ]:
from IPython.display import display, HTML

# 🎯 CẤU HÌNH FINETUNE PROMPT SIÊU TỐC:
# - FORCE_RERUN = True: luôn ép chạy lại với prompt mới (không bị skip)
# - RUN_INDICES = None: chạy cả 12 testcases (~2 phút); hoặc gán [1, 2]: chỉ test 2 shot đầu (~25 giây xem ngay)
FORCE_RERUN = True
RUN_INDICES = None

shots_to_process = test_shot_paths if 'test_shot_paths' in globals() and test_shot_paths else []
if RUN_INDICES is not None and isinstance(RUN_INDICES, list):
    shots_to_process = [p for p in shots_to_process if os.path.basename(p) in TARGET_TEST_FILENAMES and (TARGET_TEST_FILENAMES.index(os.path.basename(p)) + 1 in RUN_INDICES)]

print(f"🚀 BẮT ĐẦU CHẠY THỬ NGHIỆM FINETUNE CHO {len(shots_to_process)} TESTCASES...")
print(f"   - Output JSONL: {OUTPUT_JSONL}")
print(f"   - Chế độ: {'🔥 Chạy lại toàn bộ (Finetune Prompt mới)' if FORCE_RERUN else 'Tiếp tục từ checkpoint'}\n")

done_files = set()
if not FORCE_RERUN and os.path.exists(OUTPUT_JSONL):
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    r = json.loads(line)
                    done_files.add(r.get("shot_file"))
                except Exception:
                    pass

actual_todo = [p for p in shots_to_process if FORCE_RERUN or os.path.basename(p) not in done_files]
print(f"   - Thực tế cần chạy: {len(actual_todo)}/{len(shots_to_process)} shots\n")

if len(actual_todo) == 0:
    print("🎉 TẤT CẢ CÁC TESTCASES ĐÃ HOÀN TẤT!")
else:
    session_start_time = time.time()
    for idx, video_path in enumerate(actual_todo, 1):
        filename = os.path.basename(video_path)
        video_id, shot_idx, start_sec, end_sec = parse_shot_filename(filename)
        duration = round(end_sec - start_sec, 2)
        
        target_idx = TARGET_TEST_FILENAMES.index(filename) + 1 if filename in TARGET_TEST_FILENAMES else idx
        
        elapsed = time.time() - session_start_time
        avg_speed = elapsed / max(1, idx - 1) if idx > 1 else 15.0
        eta_mins = round(((len(actual_todo) - idx) * avg_speed) / 60, 1)
        
        # Nạp động metadata title và ASR
        video_title = get_video_title(video_id, MEDIA_INFO_DIR)
        asr_text, seg_count = get_asr_context(video_id, start_sec, end_sec, ASR_DIR)
        
        print("=" * 85)
        print(f"🔹 [TESTCASE {target_idx:02d}/12] SHOT: {filename} ({duration}s) | Tiến độ: [{idx}/{len(actual_todo)}] | ETA: ~{eta_mins}m")
        if video_title:
            print(f"  • Tiêu đề (Metadata): \"{video_title}\"")
        if asr_text:
            print(f"  • Lời thoại (ASR Input): \"{asr_text[:110]}...\" ({seg_count} segments)")
        else:
            print("  • Lời thoại (ASR Input): (Không có lời thoại)")
            
        # Dọn sạch VRAM & Reset thống kê đỉnh (Peak) VRAM trước mỗi shot
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        for d in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(d)
        
        try:
            res = caption_single_shot(
                model=model,
                processor=processor,
                video_path=video_path,
                asr_text=asr_text,
                video_title=video_title,
                max_new_tokens=MAX_NEW_TOKENS
            )
            
            # Thống kê đỉnh VRAM tiêu thụ cho shot này
            peak_gpu_details = []
            total_peak_gb = 0.0
            for d in range(torch.cuda.device_count()):
                peak_gb = round(torch.cuda.max_memory_allocated(d) / (1024**3), 2)
                peak_gpu_details.append(f"GPU {d}: {peak_gb} GB")
                total_peak_gb += peak_gb
            total_peak_gb = round(total_peak_gb, 2)
            pct_5060ti = round((total_peak_gb / 16.0) * 100, 1)
            vram_status_tag = "✅ AN TOÀN TRÊN 5060 Ti 16GB" if total_peak_gb <= 15.0 else "⚠️ NGUY CƠ TRÀN TRÊN 16GB"
            
            print(f"  ⚡ Thời gian suy luận: {res['infer_time_sec']}s | Tokens: In={res['input_tokens']}, Out={res['generated_tokens']}/512")
            print(f"  🔥 ĐỈNH VRAM TIÊU THỤ: {' | '.join(peak_gpu_details)} => Tổng: {total_peak_gb} GB ({pct_5060ti}% của RTX 5060 Ti 16GB) | {vram_status_tag}")
            print("\n  📝 CAPTION QWEN3-VL-8B SINH RA:")
            for line in res["caption"].split("\n"):
                if line.strip():
                    print(f"     {line}")
            print("=" * 85 + "\n")
            
            # Hiển thị trực quan dạng thẻ màu (Card UI) kèm thông số VRAM
            cap_formatted = res['caption'].replace('<', '&lt;').replace('>', '&gt;').replace('\n', '<br>')
            vram_badge_color = '#10b981' if total_peak_gb <= 14.5 else ('#f59e0b' if total_peak_gb <= 15.5 else '#ef4444')
            display(HTML(f'''
            <div style="background: #0f172a; border-left: 6px solid #3b82f6; padding: 18px 22px; margin: 15px 0; border-radius: 8px; color: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
                <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 10px; margin-bottom: 12px;">
                    <span style="font-size: 16px; font-weight: bold; color: #60a5fa;">🎯 TESTCASE {target_idx:02d}/12: {filename}</span>
                    <span style="background: #1e293b; color: #38bdf8; padding: 4px 10px; border-radius: 4px; font-size: 12px; font-weight: 600;">⏱️ {duration}s | ⚡ {res['infer_time_sec']}s | 🔤 In: {res['input_tokens']} tok / Out: {res['generated_tokens']} tok</span>
                </div>
                <div style="font-size: 13px; color: #94a3b8; margin-bottom: 8px;">
                    <strong style="color: #cbd5e1;">Tiêu đề:</strong> {video_title if video_title else '<i>(Trống)</i>'}<br>
                    <strong style="color: #cbd5e1;">Lời thoại ASR:</strong> {asr_text[:120] + '...' if asr_text else '<i>(Không có)</i>'}
                </div>
                <div style="margin: 10px 0; font-size: 13px; color: #cbd5e1; background: #1e293b; padding: 8px 14px; border-radius: 6px; border-left: 4px solid {vram_badge_color};">
                    <strong style="color: #f59e0b;">🔥 Đỉnh VRAM tiêu thụ:</strong> {' | '.join(peak_gpu_details)} &nbsp;|&nbsp; 
                    <strong style="color: #38bdf8;">Ước tính card đơn (RTX 5060 Ti 16GB):</strong> 
                    <span style="color: {vram_badge_color}; font-weight: bold;">{total_peak_gb} GB / 16.0 GB ({pct_5060ti}%)</span> &nbsp;—&nbsp; 
                    <span style="font-weight: 600; color: {vram_badge_color};">{vram_status_tag}</span>
                </div>
                <div style="background: #1e293b; padding: 14px 16px; border-radius: 6px; font-size: 14px; line-height: 1.6; color: #f1f5f9; border: 1px solid #475569;">
                    <strong style="color: #38bdf8; display: block; margin-bottom: 6px;">📝 KẾT QUẢ CAPTION BẢN GỐC (BFLOAT16):</strong>
                    {cap_formatted}
                </div>
            </div>
            '''))
                    
            record = {
                "test_case_idx": target_idx,
                "video_id": video_id,
                "shot_file": filename,
                "shot_idx": shot_idx,
                "shot_start_sec": start_sec,
                "shot_end_sec": end_sec,
                "duration_sec": duration,
                "video_title": video_title,
                "asr_segment_count": seg_count,
                "asr_context": asr_text,
                "caption": res["caption"],
                "model_id": MODEL_ID,
                "infer_time_sec": res["infer_time_sec"],
                "input_tokens": res["input_tokens"],
                "generated_tokens": res["generated_tokens"],
                "peak_vram_gb": total_peak_gb,
                "peak_gpu_details": peak_gpu_details,
                "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
            }
            
            with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                f.flush()
                
        except Exception as e:
            print(f"  ❌ Lỗi khi xử lý shot {filename}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
    print("\n" + "=" * 85)
    print(f"✅ ĐÃ HOÀN TẤT TOÀN BỘ 12 TESTCASES CHO QWEN3-VL-8B!")
    print(f"📁 File kết quả lưu tại: {OUTPUT_JSONL}")
    print("=" * 85)

--- 
## BƯỚC 8: Bảng Tổng Hợp & Đọc Chi Tiết Từng Caption Để Đánh Giá Finetune

In [ ]:
import os
import json

# 1. Cấu hình thư mục output và tự động dò tìm file kết quả .jsonl
OUTPUT_DIR = "/workspace/output" if os.path.exists("/workspace") else "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "qwen3vl_8b_finetuned_captions.json")

target_jsonl = None
possible_jsonls = [
    getattr(globals(), 'OUTPUT_JSONL', ''),
    os.path.join(OUTPUT_DIR, "qwen3vl_8b_finetuned_captions.jsonl"),
    "/workspace/output/qwen3vl_8b_finetuned_captions.jsonl",
    "/workspace/qwen3vl_8b_finetuned_captions.jsonl",
    "./output/qwen3vl_8b_finetuned_captions.jsonl",
    "./qwen3vl_8b_finetuned_captions.jsonl",
    "/kaggle/working/output/qwen3vl_8b_finetuned_captions.jsonl",
    "/kaggle/working/qwen3vl_8b_finetuned_captions.jsonl",
]
for p in possible_jsonls:
    if p and os.path.exists(p):
        target_jsonl = p
        break

if target_jsonl:
    records = []
    with open(target_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    
    if records:
        # Sắp xếp theo thứ tự test_case_idx
        records.sort(key=lambda r: r.get("test_case_idx", 0))
        
        # 2. XUẤT FILE JSON ĐẸP VÀO /workspace/output/
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)
        print("=" * 85)
        print(f"💾 ĐÃ XUẤT THÀNH CÔNG FILE JSON ({len(records)} shots) VÀO:")
        print(f"👉 {OUTPUT_JSON}")
        print("=" * 85 + "\n")
        
        # 3. HIỂN THỊ BẢNG TỔNG HỢP (An toàn tuyệt đối nếu chưa có pandas)
        try:
            import pandas as pd
            df = pd.DataFrame(records)
            cols_show = [c for c in ["test_case_idx", "video_id", "shot_file", "duration_sec", "infer_time_sec", "generated_tokens", "peak_vram_gb"] if c in df.columns]
            print("📊 BẢNG TỔNG KẾT HIỆU NĂNG 12 TESTCASES:")
            display(df[cols_show])
            print(f"\n📈 THỐNG KÊ TRUNG BÌNH:")
            print(f"  • Thời gian suy luận trung bình: {df['infer_time_sec'].mean():.2f} giây / shot")
            print(f"  • Số token sinh ra trung bình:  {df['generated_tokens'].mean():.1f} tokens (max=512)")
            if 'peak_vram_gb' in df.columns:
                print(f"  • Đỉnh VRAM cao nhất:         {df['peak_vram_gb'].max():.2f} GB / 16GB RTX 5060 Ti")
        except ImportError:
            print("📊 BẢNG TỔNG KẾT HIỆU NĂNG 12 TESTCASES:")
            header = f"{'STT':<4} | {'SHOT FILE':<35} | {'DUR(s)':<6} | {'TIME(s)':<7} | {'TOKENS':<7} | {'PEAK VRAM'}"
            print(header)
            print("-" * len(header))
            total_time, total_tok = 0.0, 0
            for r in records:
                vram_str = f"{r.get('peak_vram_gb', 'N/A')} GB"
                print(f"{r.get('test_case_idx', 0):<4} | {r.get('shot_file', ''):<35} | {r.get('duration_sec', 0):<6.1f} | {r.get('infer_time_sec', 0):<7.2f} | {r.get('generated_tokens', 0):<7} | {vram_str}")
                total_time += r.get('infer_time_sec', 0)
                total_tok += r.get('generated_tokens', 0)
            print(f"\n📈 Trung bình: {total_time/len(records):.2f}s/shot | {total_tok/len(records):.1f} tokens/shot")
            print("💡 (Mẹo: Có thể gõ %pip install pandas nếu muốn hiển thị dạng bảng tương tác)")
        
        print("\n" + "=" * 90)
        print("📖 ĐỌC TRỰC TIẾP TOÀN BỘ 12 CAPTION ĐÃ SINH:")
        print("=" * 90)
        for r in records:
            t_idx = r.get('test_case_idx', '?')
            print(f"\n👉 [TESTCASE #{t_idx:02d}] {r['shot_file']} ({r['duration_sec']}s)")
            if r.get('video_title'):
                print(f"   - Tiêu đề Video: \"{r['video_title']}\"")
            print(f"   - Thời gian: {r['infer_time_sec']}s | Tokens: {r['generated_tokens']}/512 | VRAM: {r.get('peak_vram_gb', 'N/A')} GB")
            print("   - CAPTION:")
            for l in r['caption'].split('\n'):
                if l.strip():
                    print(f"     {l}")
            print("-" * 70)
else:
    print(f"⚠️ Chưa tìm thấy file kết quả .jsonl tại {possible_jsonls[0]}. Hãy chạy Bước 7 trước!")
